# Cohort Discovery NLP — `/llm/*` endpoint sandbox

Exercises the experimental LLM-backed query parsing endpoints.

**Before running:** start Ollama (`ollama serve`), pull a model (`ollama pull qwen3:8b`), and start the NLP service:

```bash
uvicorn app:app --host 0.0.0.0 --port 5001
```

Wait for `CORE READY` in the log before trusting concept matches — before that the resolver runs in reduced mode (LIKE-based SQL, no synonym or acronym enrichment).


## Managing models

Ollama loads a model into memory on first use and keeps it there for `OLLAMA_KEEP_ALIVE`
(the service asks for `30m`; Ollama's own default is 5 minutes). Nothing needs loading by
hand — the first request to a model loads it — but on a 36 GB machine only one or two large
models fit at once, so knowing how to evict matters when benchmarking.

```bash
ollama list                 # what is on disk
ollama ps                   # what is in memory right now, and when it expires
ollama pull qwen3:14b       # download (resumable; re-run if it stalls)
ollama stop qwen3:8b        # evict from memory now, keep on disk
ollama rm qwen3:0.6b        # delete from disk
```

`GET /llm/models` returns both views together (`loaded: true` marks resident models), so you
can check from here rather than the shell.

Things worth knowing before you benchmark:

- **The first call to a model pays the load.** 10–30 s, and it distorts any timing you take.
  Send one throwaway query per model before you measure — `benchmark()` below does not do
  this for you.
- **Switching models evicts.** Two 8B models will not both stay resident under memory
  pressure, so an A/B loop reloads on every switch. Benchmark one model fully, then the
  next, rather than interleaving.
- **Watch for swap.** If generation drops below ~20 tok/s on Apple silicon the machine is
  usually the cause, not the model — check `sysctl vm.swapusage`. Measurements taken while
  swapping are not comparable with ones taken when not.
- **`ollama pull` exits 0 even when the download fails.** Trust `ollama list` or
  `GET /llm/models`, not the exit code.

```python
# evict everything currently resident, for a clean run
import subprocess
for m in models()["loaded"]:
    subprocess.run(["ollama", "stop", m["name"]])
```


## Test queries

Each entry is `{"query", "note", "expected"}`. The `expected` block says what the plan *should* contain, and is what section 9 scores models against — only the keys present are checked, so a case can assert one thing without over-specifying the rest.

| key | meaning |
|---|---|
| `age` | top-level demographics age `[min, max]` |
| `sex` / `race` | expected concept names |
| `death` | `"Recorded"` / `"Not recorded"` / `None` |
| `op` | `"and"` / `"or"` / `"followed_by"` |
| `n_rules` | number of top-level rule entries |
| `terms` | substrings that must appear among the search terms |
| `rule_age` | `{term substring: [min, max]}` — age carried by a **rule**, not demographics |
| `no_rule_age` | `True` means no rule may carry an age |
| `value` | `{term substring: [min, max]}` for `valueAsNumber` |
| `months` | `{term substring: n}`, or `{"*": n}` for a top-level window |
| `negated` | term substrings that must be excluded |
| `warns` | substrings expected among the warnings |

`NOT IMPLEMENTED` in a note marks something the platform cannot support regardless of parsing — BUNNY has no visit table, no "followed by", no measurement-value filters. Those queries still have an `expected` block for the part that *is* extractable.

**These expectations are my reading of your queries — correct them where you disagree.** Several are genuinely ambiguous and the note says so.


In [ ]:
SIMPLE_QUERIES = [
    {"query": "Women who were under 60 when they suffered a hip fracture",
     "note": "age belongs to the fracture, not the person",
     "expected": {"age": [0, 120], "sex": ["Female"],
                  "rule_age": {"hip fracture": [None, 60]}}},

    {"query": "Adults aged 18-30 with a diagnosis of asthma",
     "note": "ambiguous: age of person vs age at diagnosis - read as the person",
     "expected": {"age": [18, 30], "no_rule_age": True}},

    {"query": "People aged 65+ with diagnosed hypertension",
     "note": "ambiguous: age of person vs age at diagnosis - read as the person",
     "expected": {"age": [65, 120], "no_rule_age": True}},

    {"query": "Adults with type 2 diabetes diagnosed in the last 2 years",
     "note": "known miss: the age band is sometimes dropped when a window is present",
     "expected": {"age": [18, 120], "months": {"diabetes": 24}}},

    {"query": "Children under 5 with at least 3 GP visits for ear infections in 12 months",
     "note": "NOT IMPLEMENTED: visits not supported by BUNNY; occurrence counts not modelled",
     "expected": {"age": [0, 5], "warns": ["Visit-based"]}},

    {"query": "People with chronic kidney disease stage 3-5",
     "expected": {"age": [0, 120], "terms": ["chronic kidney disease"]}},

    {"query": "People with CKD", "note": "acronym",
     "expected": {"age": [0, 120], "n_rules": 1}},

    {"query": "People with Chronic Kidney Disease",
     "expected": {"age": [0, 120], "terms": ["chronic kidney disease"]}},

    {"query": "People with Renal Failure",
     "expected": {"age": [0, 120], "terms": ["renal failure"]}},

    {"query": "Adults with BMI over 30",
     "note": "numeric threshold, not part of the search term",
     "expected": {"age": [18, 120], "value": {"bmi": [30, None]}}},

    {"query": "Obese Adults",
     "note": "age word as head noun behind an adjective - was a miss, now fixed",
     "expected": {"age": [18, 120]}},

    {"query": "Adults with Obesity", "expected": {"age": [18, 120]}},

    {"query": "Adults with COPD", "note": "acronym",
     "expected": {"age": [18, 120], "no_rule_age": True}},

    {"query": "Children with COPD", "expected": {"age": [0, 17]}},

    {"query": "People with epilepsy currently under specialist neurology care",
     "note": "NOT IMPLEMENTED: neurology care is a visit",
     "expected": {"terms": ["epilepsy"], "warns": ["Visit-based"]}},

    {"query": "Women aged 18-45 with endometriosis",
     "expected": {"age": [18, 45], "sex": ["Female"], "no_rule_age": True}},

    {"query": "Men aged 50+ with benign prostatic hyperplasia",
     "expected": {"age": [50, 120], "sex": ["Male"], "no_rule_age": True}},

    {"query": "People aged 40+ with atrial fibrillation",
     "expected": {"age": [40, 120], "no_rule_age": True}},

    {"query": "Adults with heart failure", "expected": {"age": [18, 120]}},

    {"query": "People with stroke recorded in the last 5 years",
     "expected": {"age": [0, 120], "months": {"stroke": 60}}},

    {"query": "Adults with diagnosed depression", "expected": {"age": [18, 120]}},

    {"query": "Adults with severe mental illness", "expected": {"age": [18, 120]}},

    {"query": "People with schizophrenia",
     "expected": {"age": [0, 120], "terms": ["schizophrenia"]}},

    {"query": "Adults with bipolar disorder", "expected": {"age": [18, 120]}},

    {"query": "People with dementia diagnosis",
     "expected": {"age": [0, 120], "terms": ["dementia"]}},

    {"query": "Adults with rheumatoid arthritis", "expected": {"age": [18, 120]}},

    {"query": "People with inflammatory bowel disease",
     "expected": {"age": [0, 120], "terms": ["inflammatory bowel disease"]}},

    {"query": "People with Crohn's disease",
     "expected": {"age": [0, 120], "terms": ["crohn"]}},

    {"query": "People with ulcerative colitis",
     "expected": {"age": [0, 120], "terms": ["ulcerative colitis"]}},

    {"query": "Adults with a recorded allergy to penicillin",
     "expected": {"age": [18, 120], "terms": ["penicillin"]}},
]


In [ ]:
ADVANCED_QUERIES = [
    {"query": "People with cancer over the age of 50 who have been treated for hip fractures",
     "note": "the age is used once, on the person - was double-counted, now fixed",
     "expected": {"age": [50, 120], "no_rule_age": True, "n_rules": 2}},

    {"query": "Men over 60 with cancer in NHS Scotland Regions",
     "note": "NOT IMPLEMENTED: location",
     "expected": {"age": [60, 120], "sex": ["Male"], "warns": ["Location-based"]}},

    {"query": "Men over 60 with cancer in the South West",
     "note": "NOT IMPLEMENTED: location",
     "expected": {"age": [60, 120], "sex": ["Male"]}},

    {"query": "Individuals who have antibodies to SARs-CoV-2 measured after receiving two doses of any COVID-19 vaccine",
     "note": "sequencing; 'two doses' is an occurrence count and is not modelled",
     "expected": {"age": [0, 120]}},

    {"query": "Black men over the age of 60 who have had heart attacks while being on blood thinners",
     "note": "race in the demographics block",
     "expected": {"age": [60, 120], "sex": ["Male"],
                  "race": ["Black or African American"]}},

    {"query": "Adults with type 2 diabetes and at least one HbA1c result above 75 mmol/mol in the last year",
     "note": "NOT IMPLEMENTED downstream: filtering on measurement value",
     "expected": {"age": [18, 120], "value": {"hba1c": [75, None]}}},

    {"query": "People with chronic kidney disease and an eGFR below 45 on two occasions at least 90 days apart",
     "note": "NOT IMPLEMENTED downstream: measurement value; 'two occasions 90 days apart' not modelled",
     "expected": {"value": {"egfr": [None, 45]}}},

    {"query": "Adults with hypertension who are not currently prescribed any antihypertensive medication",
     "note": "negative queries are a poor fit for cohort discovery - absence of a record is not absence of treatment",
     "expected": {"age": [18, 120], "negated": ["antihypertensive"]}},

    {"query": "People with asthma who have had 2+ oral steroid courses in the last 12 months",
     "note": "NOT IMPLEMENTED: multiple occurrences",
     "expected": {"months": {"steroid": 12}}},

    {"query": "People with asthma who have had oral steroid courses in the last 12 months",
     "note": "the supported reduction of the query above",
     "expected": {"months": {"steroid": 12}, "n_rules": 2}},

    {"query": "Adults with COPD who started home oxygen therapy in the last 2 years",
     "expected": {"age": [18, 120], "months": {"oxygen": 24}}},

    {"query": "People with atrial fibrillation who have had an ischaemic stroke despite being prescribed anticoagulation",
     "expected": {"age": [0, 120], "n_rules": 3}},

    {"query": "Women with polycystic ovary syndrome who later developed type 2 diabetes",
     "note": "sequencing - the schema supports followed_by even though BUNNY cannot run it",
     "expected": {"sex": ["Female"], "op": "followed_by"}},

    {"query": "Pregnant women with gestational diabetes who delivered preterm",
     "expected": {"sex": ["Female"]}},

    {"query": "Pregnant women with gestational diabetes who delivered before 37 weeks",
     "note": "'before 37 weeks' is gestational age, not patient age - must not become an ageConstraint",
     "expected": {"sex": ["Female"], "no_rule_age": True}},

    {"query": "Children diagnosed with autism who also have epilepsy",
     "expected": {"age": [0, 17], "op": "and", "n_rules": 2}},

    {"query": "People with depression who started an SSRI and then switched antidepressant within 6 months",
     "note": "SSRI is a drug-class acronym; sequencing",
     "expected": {"op": "followed_by"}},

    {"query": "Adults prescribed long-term opioids (90+ consecutive days) for non-cancer pain",
     "note": "NOT IMPLEMENTED: consecutive days; 'long-term' is the user's own definition",
     "expected": {"age": [18, 120], "no_rule_age": True}},

    {"query": "People with osteoarthritis who received a knee replacement and had a postoperative infection within 90 days",
     "note": "sequencing",
     "expected": {"age": [0, 120]}},

    {"query": "Adults with a new diagnosis of heart failure who had no recorded hypertension beforehand",
     "note": "NOT IMPLEMENTED: BUNNY cannot filter 'followed by'",
     "expected": {"age": [18, 120], "negated": ["hypertension"]}},

    {"query": "People with colorectal cancer who received chemotherapy and later developed neutropenia requiring hospital admission",
     "note": "NOT IMPLEMENTED: 'followed by'; admission is a visit",
     "expected": {"op": "followed_by"}},

    {"query": "Adults with breast cancer who later developed cardiomyopathy after anthracycline treatment",
     "note": "NOT IMPLEMENTED: 'followed by'",
     "expected": {"age": [18, 120], "op": "followed_by"}},

    {"query": "People with long COVID codes and persistent breathlessness recorded 12+ weeks after acute infection",
     "note": "NOT IMPLEMENTED: 'followed by'",
     "expected": {"age": [0, 120]}},

    {"query": "Adults admitted with pneumonia who were not vaccinated against influenza that season",
     "note": "NOT IMPLEMENTED: 'followed by'; admission is a visit",
     "expected": {"age": [18, 120], "negated": ["influenza"]}},

    {"query": "People with type 2 diabetes and COPD, and all-cause mortality within 5 years of COPD diagnosis",
     "note": "NOT IMPLEMENTED: 'followed by'. Mortality belongs in demographics, not as a search term",
     "expected": {"death": "Recorded"}},

    {"query": "Adults with severe obesity who underwent bariatric surgery and had type 2 diabetes remission within 12 months",
     "note": "sequencing",
     "expected": {"age": [18, 120]}},

    {"query": "People with migraine who started a CGRP inhibitor and had reduced emergency attendances afterward",
     "note": "NOT IMPLEMENTED: visits. CGRP is a drug-class acronym",
     "expected": {"age": [0, 120]}},

    {"query": "Adults with psoriasis on systemic therapy who developed serious infection requiring hospitalisation",
     "note": "NOT IMPLEMENTED: visits",
     "expected": {"age": [18, 120], "warns": ["Visit-based"]}},
]


In [ ]:
ALL_QUERIES = SIMPLE_QUERIES + ADVANCED_QUERIES
len(SIMPLE_QUERIES), len(ADVANCED_QUERIES), len(ALL_QUERIES)


## Setup


In [ ]:
import json
import time
from collections import Counter

import httpx
import pandas as pd

BASE = "http://localhost:5001"


def parse(query, fill_concepts=True, model=None, **params):
    """POST /llm/parse. Raises with the server's detail message on error."""
    body = {"query": query, "fill_concepts": fill_concepts}
    if model:
        body["model"] = model
    r = httpx.post(f"{BASE}/llm/parse", json=body, params=params, timeout=300)
    if r.status_code != 200:
        raise RuntimeError(f"HTTP {r.status_code}: {r.text}")
    return r.json()


def models():
    """GET /llm/models."""
    r = httpx.get(f"{BASE}/llm/models", timeout=30)
    r.raise_for_status()
    return r.json()


## 1. Which models are available?


In [ ]:
info = models()
print(f"default: {info['default']}   total: {info['total']}")
pd.DataFrame(info["models"])


In [ ]:
# currently resident in memory, and when they expire
pd.DataFrame(info["loaded"])


## 2. A single query, end to end

`plan` is the raw model output; `tree` is the query-builder JSON it converts to. When something looks wrong, `plan` tells you whether the model or the conversion is at fault.


In [ ]:
result = parse("adults with cancer or diabetes")
print(json.dumps(result["plan"], indent=2))
print("timings:", result["duration_ms"])


In [ ]:
result["tree"]["demographics"]


### Structure only

`fill_concepts=False` skips the database entirely — every `rule.concept` stays `null`. Use it when judging structure rather than concept quality; it is much faster.


In [ ]:
blank = parse("adults with cancer or diabetes", fill_concepts=False)
print(json.dumps(blank["tree"], indent=2))


## 3. Readable tree printer


In [ ]:
def show(tree, indent=0):
    """Print the rule tree, one node per line."""
    for node in tree.get("rules", []):
        pad = "  " * indent
        if "combinator" in node:
            print(f"{pad}-- {node['combinator'].upper()} --")
        elif "rules" in node:
            print(f"{pad}GROUP")
            show(node, indent + 1)
        else:
            concept = node["rule"]["concept"]
            if concept is None:
                target = "(blank)"
            elif concept["concept_id"] is None:
                target = f"UNMATCHED {concept['name']!r}"
            else:
                n_alt = len(concept.get("alternatives", []))
                target = f"[{concept['concept_id']}] {concept['name']}  (+{n_alt} alts)"
            extras = []
            if node.get("exclude"):
                extras.append("EXCLUDED")
            for key in ("ageConstraint", "valueAsNumber", "timeConstraint"):
                if key in node:
                    value = node[key]
                    if key == "timeConstraint":
                        value = [v[:10] if v else None for v in value]
                    extras.append(f"{key}={value}")
            suffix = f"   {' '.join(extras)}" if extras else ""
            print(f"{pad}{node['searchTerm']!r} -> {target}{suffix}")


def summarise(result):
    demo = result["tree"]["demographics"]
    death = demo["death"]["label"] if demo["death"] else "-"
    sex = [c["name"] for c in demo["sex"]] or "-"
    race = [c["name"] for c in demo["race"]] or "-"
    print(f"age={demo['age']}  sex={sex}  race={race}  death={death}")
    show(result["tree"])
    for w in result["warnings"]:
        print(f"  ! {w}")


In [ ]:
summarise(parse("women who were under 60 when they suffered a hip fracture"))


## 4. Current age vs age at the event

The two readings give different cohorts. A 70-year-old who broke a hip at 55 matches the first query but not the second. Check the age lands in the right place — `demographics.age` for the person, `ageConstraint` on the rule for the event.


In [ ]:
for q in [
    "women who were under 60 when they suffered a hip fracture",
    "women aged 18-45 with endometriosis",
    "patients diagnosed with diabetes before the age of 40",
    "people aged 65+ with hypertension",
]:
    print(f"\n=== {q}")
    summarise(parse(q, fill_concepts=False))


## 5. Run the whole set

Structure only, so no database work. Expect roughly 3-6 s per query — budget a few minutes.


In [ ]:
def run_all(queries, fill_concepts=False, model=None):
    rows = []
    for i, case in enumerate(queries, 1):
        query, note = case["query"], case.get("note", "")
        try:
            t0 = time.monotonic()
            result = parse(query, fill_concepts=fill_concepts, model=model)
            secs = time.monotonic() - t0
            demo = result["tree"]["demographics"]
            plan = result["plan"]
            rows.append({
                "query": query,
                "note": note,
                "age": demo["age"],
                "sex": [c["name"] for c in demo["sex"]],
                "race": [c["name"] for c in demo["race"]],
                "death": demo["death"]["label"] if demo["death"] else None,
                "op": plan.get("op"),
                "terms": [r.get("term") or r.get("terms") for r in plan.get("rules", [])],
                "n_warnings": len(result["warnings"]),
                "secs": round(secs, 1),
                "plan": plan,
                "tree": result["tree"],
                "warnings": result["warnings"],
            })
        except Exception as e:
            rows.append({"query": query, "note": note, "error": str(e)})
        print(f"{i}/{len(queries)}", end="\r")
    return pd.DataFrame(rows)


results = run_all(SIMPLE_QUERIES)
results[["query", "age", "sex", "death", "op", "terms", "secs"]]


In [ ]:
advanced = run_all(ADVANCED_QUERIES)
advanced[["query", "age", "sex", "race", "op", "terms", "n_warnings", "secs"]]


## 6. Where does it disagree with you?

Some slices for spotting bad parses.


In [ ]:
everything = pd.concat([results, advanced], ignore_index=True)

# any term that still looks like a time window, a number or a demographic
leaked = everything[everything["terms"].astype(str).str.contains(
    r"year|month|week|age|adult|child|male|female|died|death", case=False, na=False
)]
leaked[["query", "terms"]]


In [ ]:
# queries where the model reached for followed_by
everything[everything["op"] == "followed_by"][["query", "terms"]]


In [ ]:
# every warning raised, most common first
counter = Counter(w for ws in everything["warnings"].dropna() for w in ws)
for warning, n in counter.most_common():
    print(f"{n:3d}  {warning}")


In [ ]:
# inspect any single row in full
row = everything.iloc[0]
print(row["query"])
print(json.dumps(row["plan"], indent=2))


## 7. Compare models

`model` is a per-request override, so no restart is needed. Pull the other model first (`ollama pull qwen3:14b`).


In [ ]:
# available = [m["name"] for m in models()["models"]]
# comparison = {name: run_all(SIMPLE_QUERIES[:5], model=name) for name in available}
# comparison["qwen3:8b"][["query", "age", "terms", "secs"]]


## 8. Concept resolution

With `fill_concepts=True` each blank concept is filled by the existing resolver — the same `resolver.search()` call `/extract` makes. The best match goes in `rule.concept`, the rest into `concept.alternatives`.

Concept *quality* is the existing resolver's behaviour, not the LLM's. Compare against `/extract` below before blaming the model.


In [ ]:
resolved = parse("adults with cancer or diabetes")
summarise(resolved)


In [ ]:
# the same query through the existing pipeline, for comparison
r = httpx.post(f"{BASE}/extract", json={"query": "adults with cancer or diabetes"}, timeout=120)
extract = r.json()
pd.DataFrame([
    {"text": e["text"],
     "concept_id": e["attributes"]["concept_id"],
     "concept_name": e["attributes"]["concept_name"],
     "domain": e["attributes"]["domain_id"]}
    for e in extract["entities"]
]).head(15)


## 9. Benchmarking models

Timing alone will not choose a model: one that is fast but puts the age in the wrong place
is worse than a slower one that gets it right. This scores models against the `expected`
blocks defined at the top of the notebook, so rank on accuracy first and use latency to
break ties.

Queries whose `expected` block is empty are skipped. Edit the expectations up there rather
than here — there is only one list.


In [ ]:
def _terms(plan):
    return [(r.get("term") or "", r) for r in plan.get("rules", [])]


def _find(plan, needle):
    for term, rule in _terms(plan):
        if needle.lower() in term.lower():
            return rule
    return None


def check(case, result):
    """Return [(check_name, passed, detail)] for one `expected` block."""
    plan = result["plan"]
    demo = result["tree"]["demographics"]
    out = []

    def add(name, passed, detail=""):
        out.append((name, bool(passed), detail))

    if "age" in case:
        add("age", demo["age"] == case["age"], f"got {demo['age']}")
    if "sex" in case:
        got = [c["name"] for c in demo["sex"]]
        add("sex", got == case["sex"], f"got {got}")
    if "race" in case:
        got = [c["name"] for c in demo["race"]]
        add("race", got == case["race"], f"got {got}")
    if "death" in case:
        got = demo["death"]["label"] if demo["death"] else None
        add("death", got == case["death"], f"got {got}")
    if "op" in case:
        add("op", plan.get("op") == case["op"], f"got {plan.get('op')}")
    if "n_rules" in case:
        got = len(plan.get("rules", []))
        add("n_rules", got == case["n_rules"], f"got {got}")
    if case.get("no_rule_age"):
        offenders = [t for t, r in _terms(plan)
                     if r.get("age_min") is not None or r.get("age_max") is not None]
        add("no_rule_age", not offenders, f"age on {offenders}")
    for needle, (lo, hi) in case.get("rule_age", {}).items():
        rule = _find(plan, needle)
        got = (rule or {}).get("age_min"), (rule or {}).get("age_max")
        add(f"rule_age[{needle}]", rule is not None and got == (lo, hi), f"got {got}")
    for needle, (lo, hi) in case.get("value", {}).items():
        rule = _find(plan, needle)
        got = (rule or {}).get("value_min"), (rule or {}).get("value_max")
        add(f"value[{needle}]", rule is not None and got == (lo, hi), f"got {got}")
    for needle, months in case.get("months", {}).items():
        if needle == "*":
            got = plan.get("last_months")
        else:
            got = (_find(plan, needle) or {}).get("last_months")
        add(f"months[{needle}]", got == months, f"got {got}")
    for needle in case.get("terms", []):
        add(f"terms[{needle}]", _find(plan, needle) is not None,
            f"got {[t for t, _ in _terms(plan)]}")
    for needle in case.get("warns", []):
        got = result.get("warnings", [])
        add(f"warns[{needle}]", any(needle.lower() in w.lower() for w in got),
            f"got {len(got)} warnings")
    for needle in case.get("negated", []):
        rule = _find(plan, needle)
        add(f"negated[{needle}]", bool(rule and rule.get("not")), f"got {rule}")

    return out


def benchmark(model_names, cases=None, repeats=1):
    """Score each model against the `expected` blocks. Returns (summary, detail) DataFrames."""
    cases = [c for c in (cases or ALL_QUERIES) if c.get("expected")]
    rows, detail = [], []
    for name in model_names:
        passed = total = 0
        secs = []
        failures = 0
        for case in cases:
            for _ in range(repeats):
                try:
                    t0 = time.monotonic()
                    result = parse(case["query"], fill_concepts=False, model=name)
                    secs.append(time.monotonic() - t0)
                    for check_name, ok, note in check(case["expected"], result):
                        total += 1
                        passed += ok
                        detail.append({"model": name, "query": case["query"],
                                       "check": check_name, "pass": ok, "detail": note})
                except Exception as e:
                    failures += 1
                    detail.append({"model": name, "query": case["query"],
                                   "check": "ERROR", "pass": False, "detail": str(e)[:120]})
            print(f"{name}: {total} checks", end="\r")
        rows.append({
            "model": name,
            "checks": total,
            "passed": passed,
            "accuracy": round(passed / total, 3) if total else 0.0,
            "median_secs": round(pd.Series(secs).median(), 2) if secs else None,
            "slowest_secs": round(max(secs), 2) if secs else None,
            "errors": failures,
        })
    return (pd.DataFrame(rows).sort_values("accuracy", ascending=False),
            pd.DataFrame(detail))


In [ ]:
# Every model you have pulled. Drop names from this list to keep the run short:
# each model is ~21 checks and reloads into memory on first use.
candidates = [m["name"] for m in models()["models"]]
print(candidates)


In [ ]:
summary, detail = benchmark(candidates)
summary


In [ ]:
# what each model got wrong
failed = detail[~detail["pass"]]
failed.pivot_table(index=["query", "check"], columns="model",
                   values="detail", aggfunc="first").fillna("ok")


In [ ]:
# which checks are hardest across all models
detail.groupby("check")["pass"].mean().sort_values().head(12)
